# Übung 02 — Datenanalyse mit pandas (Lösung)

Bezug: *Factsheet Data Science*, Kapitel 1, 6 und 12 · *Python & SQL Referenz*, Abschnitt 1.7 und Teil 3.

Ein Sportverein möchte wissen, wie oft seine Mitglieder ins Training kommen. Die Daten liegen in
`Data/teilnahmen.csv` (eine Zeile pro Teilnahme) und `Data/mitglieder.csv` (eine Zeile pro Mitglied).

Ergänze die Zellen mit `# TODO`. Die Zellen **Selbstkontrolle** prüfen dein Ergebnis.

## Bibliotheken

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# Ordner für Ergebnisse (wird von Git ignoriert)
os.makedirs("output", exist_ok=True)

## 2.1 Daten laden und prüfen
Lies `Data/teilnahmen.csv` in den DataFrame `teilnahmen` ein. Zeige die ersten 5 Zeilen, die
Spalteninformationen (`info()`) und die Grösse (`shape`) an.

In [ ]:
teilnahmen = pd.read_csv("Data/teilnahmen.csv")

print(teilnahmen.shape)
teilnahmen.info()
teilnahmen.head()

In [ ]:
# Selbstkontrolle 2.1
assert teilnahmen.shape == (15, 3)
assert list(teilnahmen.columns) == ["name", "team", "datum"]
print("2.1 richtig ✔")

## 2.2 Trainings pro Person
Zähle mit `groupby` die Trainings pro Person und sortiere absteigend. Ergebnis: Series `pro_person`.

In [ ]:
pro_person = teilnahmen.groupby("name")["datum"].count().sort_values(ascending=False)
pro_person

In [ ]:
# Selbstkontrolle 2.2
assert pro_person.to_dict() == {"Anna": 5, "Lea": 4, "Luca": 3, "Mia": 2, "Jonas": 1}
assert pro_person.index[0] == "Anna"
print("2.2 richtig ✔")

## 2.3 Wer kommt selten?
Filtere `pro_person` mit einer booleschen Maske: Wer hat **weniger als 3** Trainings?
Speichere die Namen alphabetisch sortiert als Liste in `selten`.

*Frage zum Nachdenken:* Noah ist Mitglied, taucht hier aber nicht auf. Warum? (→ Aufgabe 2.6)

In [ ]:
selten = sorted(pro_person[pro_person < 3].index)
selten
# Noah fehlt, weil er keine einzige Teilnahme hat — er kommt in teilnahmen.csv gar nicht vor.

In [ ]:
# Selbstkontrolle 2.3
assert selten == ["Jonas", "Mia"]
print("2.3 richtig ✔")

## 2.4 Teilnahmen pro Team
Wie viele Teilnahmen hat jedes Team? Ergebnis: Series `pro_team`, absteigend sortiert.

In [ ]:
pro_team = teilnahmen.groupby("team")["name"].count().sort_values(ascending=False)
pro_team

In [ ]:
# Selbstkontrolle 2.4
assert pro_team.to_dict() == {"U15": 9, "Damen 1": 6}
print("2.4 richtig ✔")

## 2.5 Mit Datumswerten arbeiten
1. Wandle die Spalte `datum` mit `pd.to_datetime` in echte Datumswerte um.
2. Ergänze eine Spalte `monat` mit der Monatszahl (`.dt.month`).
3. Wie viele Teilnahmen gab es im **September** (Monat 9)? → `anzahl_september`
4. Erstelle eine Tabelle Teams × Monate mit der Anzahl Teilnahmen (`groupby([...]).size().unstack()`).

In [ ]:
teilnahmen["datum"] = pd.to_datetime(teilnahmen["datum"])
teilnahmen["monat"] = teilnahmen["datum"].dt.month

anzahl_september = len(teilnahmen[teilnahmen["monat"] == 9])
team_monat = teilnahmen.groupby(["team", "monat"]).size().unstack(fill_value=0)
team_monat

In [ ]:
# Selbstkontrolle 2.5
assert pd.api.types.is_datetime64_any_dtype(teilnahmen["datum"])
assert anzahl_september == 12
assert team_monat.loc["U15", 9] == 7 and team_monat.loc["Damen 1", 10] == 1
print("2.5 richtig ✔")

## 2.6 Tabellen verbinden (merge = SQL-JOIN)
Lies `Data/mitglieder.csv` in `mitglieder` ein. Verbinde die Mitglieder mit `pro_person` so, dass
**alle Mitglieder** erscheinen (auch Noah) — fehlende Werte werden 0.

Tipp: `pro_person.rename("anzahl").reset_index()` macht aus der Series einen DataFrame mit den Spalten
`name` und `anzahl`. Danach `merge(..., on="name", how="left")` und `fillna(0)`.
Ergebnis: DataFrame `uebersicht` mit den Spalten `name`, `team`, `anzahl` (als ganze Zahl).

In [ ]:
mitglieder = pd.read_csv("Data/mitglieder.csv")

anzahl = pro_person.rename("anzahl").reset_index()
uebersicht = mitglieder.merge(anzahl, on="name", how="left")
uebersicht["anzahl"] = uebersicht["anzahl"].fillna(0).astype(int)
uebersicht = uebersicht[["name", "team", "anzahl"]].sort_values("anzahl", ascending=False)
uebersicht

In [ ]:
# Selbstkontrolle 2.6
assert len(uebersicht) == 6
assert dict(zip(uebersicht["name"], uebersicht["anzahl"]))["Noah"] == 0
assert list(uebersicht.columns) == ["name", "team", "anzahl"]
print("2.6 richtig ✔")

## 2.7 SQL in pandas übersetzen
Übersetze diese SQL-Abfrage in pandas (Ergebnis: DataFrame `neu`):
```sql
SELECT name, eintritt
FROM mitglieder
WHERE eintritt >= 2021
ORDER BY eintritt;
```

In [ ]:
neu = mitglieder[mitglieder["eintritt"] >= 2021][["name", "eintritt"]].sort_values("eintritt")
neu

In [ ]:
# Selbstkontrolle 2.7
assert list(neu["name"]) == ["Lea", "Anna", "Luca", "Jonas"]
assert list(neu.columns) == ["name", "eintritt"]
print("2.7 richtig ✔")

## 2.8 Visualisieren und exportieren
1. Zeichne ein Balkendiagramm der Teilnahmen pro Team (`pro_team.plot(kind="bar", ...)`) mit Titel und
   y-Achsenbeschriftung und speichere es als `output/teilnahmen_pro_team.png`.
2. Speichere `uebersicht` als `output/uebersicht.csv` so, dass Excel mit Schweizer Einstellungen die
   Datei direkt korrekt öffnet (Semikolon, `utf-8-sig`, ohne Index).

In [ ]:
pro_team.plot(kind="bar", title="Trainingsteilnahmen pro Team", color="#5DADE2")
plt.ylabel("Anzahl Teilnahmen")
plt.xticks(rotation=0)
plt.savefig("output/teilnahmen_pro_team.png", dpi=150, bbox_inches="tight")
plt.show()

uebersicht.to_csv("output/uebersicht.csv", sep=";", index=False, encoding="utf-8-sig")

In [ ]:
# Selbstkontrolle 2.8
assert os.path.exists("output/teilnahmen_pro_team.png")
kontrolle = pd.read_csv("output/uebersicht.csv", sep=";", encoding="utf-8-sig")
assert kontrolle.shape == (6, 3)
print("2.8 richtig ✔")

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
import os
import platform
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')